# features

In [5]:
from scipy.signal import welch, butter, filtfilt

bands = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta":  (13, 30),
    "gamma": (30, 40)
}

def compute_psd_batch(windows, sfreq):
    freqs, psd = welch(
        windows,
        fs=sfreq,
        nperseg=int(sfreq*2),
        axis=2
    )
    return freqs, psd

def bandpower_from_psd(freqs, psd):
    total_power = np.trapezoid(psd, freqs, axis=2)
    feats = []

    for low, high in bands.values():
        idx = (freqs >= low) & (freqs <= high)
        band_power = np.trapezoid(psd[:, :, idx], freqs[idx], axis=2)
        feats.append(band_power / (total_power + 1e-10))

    return np.concatenate(feats, axis=1)

def bandpass(data, sfreq, low, high):
    b, a = butter(4, [low/(sfreq/2), high/(sfreq/2)], btype='band')
    return filtfilt(b, a, data, axis=1)

def fast_connectivity(window):
    corr = np.corrcoef(window)
    return corr[np.triu_indices_from(corr, k=1)]

def coherence_band_fast(window, sfreq, band):
    filtered = bandpass(window, sfreq, band[0], band[1])
    return fast_connectivity(filtered)

def fast_band_connectivity(windows, sfreq, band):
    b, a = butter(4, [band[0]/(sfreq/2), band[1]/(sfreq/2)], btype='band')

    feats = []
    for w in windows:
        f = filtfilt(b, a, w, axis=1)
        corr = np.corrcoef(f)
        feats.append(corr[np.triu_indices_from(corr, k=1)])

    return np.array(feats)

def spectral_entropy(psd):
    psd_norm = psd / (np.sum(psd, axis=-1, keepdims=True) + 1e-10)
    return -np.sum(psd_norm * np.log(psd_norm + 1e-10), axis=-1)

def hjorth_params(signal):
    diff1 = np.diff(signal)
    diff2 = np.diff(diff1)

    var0 = np.var(signal)
    var1 = np.var(diff1)
    var2 = np.var(diff2)

    activity = var0
    mobility = np.sqrt(var1 / (var0 + 1e-10))
    complexity = np.sqrt(var2 / (var1 + 1e-10)) / (mobility + 1e-10)

    return np.array([activity, mobility, complexity])


def extract_features_fast(windows, sfreq):
    """
    windows: array (n_windows, n_channels, n_times)
    raw_info: MNE Raw.info
    sfreq: sampling frequency
    """
    # --- Precompute PSD for all windows ---
    freqs, psd = compute_psd_batch(windows, sfreq)  # shape: (n_windows, n_channels, n_freqs)

    # --- Bandpower features ---
    bp = bandpower_from_psd(freqs, psd)  # returns (n_windows, n_channels * n_bands)

    # --- Spectral entropy per channel ---
    ent = spectral_entropy(psd)  # (n_windows, n_channels)

    # --- Connectivity for all bands ---
    bands_list = {
        "delta": (1,4),
        "theta": (4,8),
        "alpha": (8,13),
        "beta":  (13,30),
        "gamma": (30,40)
    }

    # conn_features = []
    # for band_name, band_range in bands_list.items():
    #     # Compute coherence for this band for each window
    #     conn_band = np.array([coherence_band_fast(w, sfreq, band_range) for w in windows])
    #     conn_features.append(conn_band)
    # conn_all = np.concatenate(conn_features, axis=1)  # shape: (n_windows, n_conn * n_bands)

    conn_features = []
    for band_name, band_range in bands_list.items():
        conn_features.append(fast_band_connectivity(windows, sfreq, band_range))

    conn_all = np.concatenate(conn_features, axis=1)

    # --- Hjorth parameters per channel ---
    hjorth_feats = np.array([
        np.concatenate([hjorth_params(ch) for ch in w])
        for w in windows
    ])  # shape: (n_windows, n_channels*3)

    X = np.concatenate([bp, ent, conn_all, hjorth_feats], axis=1)
    return X

# features 2

In [2]:
def extract_features_fast(windows, sfreq):
    freqs, psd = compute_psd_batch(windows, sfreq)

    bp = bandpower_from_psd(freqs, psd)
    ent = spectral_entropy(psd)

    bands_list = {
        "delta": (1,4),
        "theta": (4,8),
        "alpha": (8,13),
        "beta":  (13,30),
        "gamma": (30,40)
    }

    conn_features = []
    for band_name, band_range in bands_list.items():
        conn_features.append(fast_band_connectivity(windows, sfreq, band_range))

    conn_all = np.concatenate(conn_features, axis=1)

    hjorth_feats = np.array([
        np.concatenate([hjorth_params(ch) for ch in w])
        for w in windows
    ])

    X = np.concatenate([bp, ent, conn_all, hjorth_feats], axis=1)

    # ---- feature block sizes ----
    feature_info = {
        "bandpower": bp.shape[1],
        "entropy": ent.shape[1],
        "connectivity": conn_all.shape[1],
        "hjorth": hjorth_feats.shape[1]
    }

    return X, feature_info

# features (no conn)

In [3]:
def extract_features_fast(windows, sfreq):
    freqs, psd = compute_psd_batch(windows, sfreq)

    # -------------------------
    # 1. Bandpower ratios (KEEP)
    # -------------------------
    bp = bandpower_from_psd(freqs, psd)

    # -------------------------
    # 2. Spectral entropy (KEEP)
    # -------------------------
    ent = spectral_entropy(psd)

    # -------------------------
    # 3. Log-variance (REPLACES Hjorth)
    # -------------------------
    # more stable than derivatives
    log_var = np.log(np.var(windows, axis=2) + 1e-10)

    # -------------------------
    # 4. (Optional lightweight temporal stability feature)
    # -------------------------
    # mean absolute signal per channel
    mean_abs = np.mean(np.abs(windows), axis=2)

    # -------------------------
    # Final feature matrix
    # -------------------------
    X = np.concatenate([bp, ent, log_var, mean_abs], axis=1)

    # -------------------------
    # Feature block sizes
    # -------------------------
    feature_info = {
        "bandpower": bp.shape[1],
        "entropy": ent.shape[1],
        "log_var": log_var.shape[1],
        "mean_abs": mean_abs.shape[1]
    }

    return X, feature_info

# window function

In [4]:
def window_raw(raw, label):
    raw = raw.copy()
    raw.resample(125)

    sfreq = raw.info["sfreq"]
    win = int(WINDOW_SEC * sfreq)

    data = raw.get_data()

    X, y = [], []

    for start in range(0, data.shape[1] - win, win):
        x = data[:, start:start + win]

        # x = (x - x.mean(axis=1, keepdims=True)) / (
        #     x.std(axis=1, keepdims=True) + 1e-10
        # )

        X.append(x)
        y.append(label)

    return X, y

# datasets

## DS006848

In [ ]:
import numpy as np
from eegdash.dataset import DS003810   # contains resting + task
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier

WINDOW_SEC = 1.0

def window_raw(raw, label):
    raw = raw.copy()
    raw.resample(125)

    sfreq = raw.info["sfreq"]
    win = int(WINDOW_SEC * sfreq)
    data = raw.get_data()

    X, y = [], []
    for start in range(0, data.shape[1] - win, win):
        x = data[:, start:start + win]
        X.append(x)
        y.append(label)

    return X, y


def extract_features(X):
    """Simple features: mean/std/bandpower-like proxy"""
    X = np.array(X)  # (n_samples, n_channels, n_time)

    mean = X.mean(axis=2)
    std = X.std(axis=2)
    ptp = np.ptp(X, axis=2)

    feats = np.concatenate([mean, std, ptp], axis=1)
    return feats


dataset = DS003810(cache_dir="./data")

subjects = dataset.subjects
subjects = sorted(subjects)

# split by subject
train_subj, test_subj = train_test_split(subjects, test_size=0.5, random_state=42)

X_train, y_train = [], []
X_test, y_test = [], []

def process_subject(subj, target_X, target_y):
    sessions = dataset.sessions(subj)

    for sess in sessions:
        runs = dataset.runs(subj, sess)

        for run in runs:
            raw, events = dataset.load_run(subj, sess, run)

            # adjust labels depending on dataset annotation
            for label_name, label_value in [
                ("rest", 0),
                ("task", 1),
            ]:
                if label_name in events:
                    for onset in events[label_name]:
                        cropped = raw.copy().crop(onset, onset + 60)  # 60s segment
                        X, y = window_raw(cropped, label_value)
                        target_X.extend(X)
                        target_y.extend(y)


for subj in train_subj:
    process_subject(subj, X_train, y_train)

for subj in test_subj:
    process_subject(subj, X_test, y_test)

X_train_feat = extract_features(X_train)
X_test_feat = extract_features(X_test)

y_train = np.array(y_train)
y_test = np.array(y_test)

model = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    n_jobs=-1
)

model.fit(X_train_feat, y_train)

preds = model.predict(X_test_feat)
probs = model.predict_proba(X_test_feat)[:, 1]

print(classification_report(y_test, preds))
print("ROC AUC:", roc_auc_score(y_test, probs))

In [4]:
import numpy as np
from eegdash.dataset import DS006848

WINDOW_SEC = 0.25
KEEP_CHANNELS = 15

dataset = DS006848(cache_dir="./data")

all_X = []
all_y = []

for rec in dataset.datasets[:5]:
    raw = rec.raw
    raw.load_data()
    raw.pick(raw.ch_names[:KEEP_CHANNELS])

    name = str(raw.filenames[0]).lower()

    if "task-rest" in name:
        label = 0
    elif "task-verbalwm" in name:
        label = 1
    else:
        continue

    X, y = window_raw(raw, label)

    all_X.extend(X)
    all_y.extend(y)

X_DS006848 = np.array(all_X)
y_DS006848 = np.array(all_y)

print(X_DS006848.shape, np.bincount(y_DS006848))

[04/20/26 15:35:35] WARNING  File not found on S3, skipping:                                      ]8;id=189771;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/downloader.py\downloader.py]8;;\:]8;id=986401;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/downloader.py#146\146]8;;\
                             s3://openneuro.org/ds006848/sub-021/eeg/sub-021_task-rest_eeg.dat                     

[04/20/26 15:35:39] WARNING  Companion file .dat not found on S3 for                                    ]8;id=586659;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/dataset/base.py\base.py]8;;\:]8;id=973009;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/dataset/base.py#483\483]8;;\
                             s3://openneuro.org/ds006848/sub-021/eeg/sub-021_task-rest_eeg.vhdr                    

Reading 0 ... 753799  =      0.000 ...   753.799 secs...


[04/20/26 15:35:44] WARNING  File not found on S3, skipping:                                      ]8;id=43681;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/downloader.py\downloader.py]8;;\:]8;id=653720;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/downloader.py#146\146]8;;\
                             s3://openneuro.org/ds006848/sub-021/eeg/sub-021_task-verbalwm_eeg.da                  
                             t                                                                                     

[04/20/26 15:35:49] WARNING  Companion file .dat not found on S3 for                                    ]8;id=443542;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/dataset/base.py\base.py]8;;\:]8;id=216882;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/dataset/base.py#483\483]8;;\
                             s3://openneuro.org/ds006848/sub-021/eeg/sub-021_task-verbalwm_eeg.vhdr                

...

[04/20/26 15:36:08] WARNING  Companion file .dat not found on S3 for                                    ]8;id=862352;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/dataset/base.py\base.py]8;;\:]8;id=197721;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/dataset/base.py#483\483]8;;\
                             s3://openneuro.org/ds006848/sub-026/eeg/sub-026_task-rest_eeg.vhdr                    

Reading 0 ... 735339  =      0.000 ...   735.339 secs...


[04/20/26 15:36:11] WARNING  File not found on S3, skipping:                                      ]8;id=474456;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/downloader.py\downloader.py]8;;\:]8;id=140250;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/downloader.py#146\146]8;;\
                             s3://openneuro.org/ds006848/sub-010/eeg/sub-010_task-verbalwm_eeg.da                  
                             t                                                                                     

[04/20/26 15:36:12] WARNING  Companion file .dat not found on S3 for                                    ]8;id=489155;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/dataset/base.py\base.py]8;;\:]8;id=891263;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/dataset/base.py#483\483]8;;\
                             s3://openneuro.org/ds006848/sub-010/eeg/sub-010_task-verbalwm_eeg.vhdr                

Reading 0 ... 4888899  =      0.000 ...  4888.899 secs...
(69911, 15, 31) [ 6004 63907]


## DS003810

In [16]:
import numpy as np
from eegdash.dataset import DS003810

WINDOW_SEC = 0.25
KEEP_CHANNELS = 15

dataset = DS003810(cache_dir="./data")


all_X, all_y = [], []

for rec in dataset.datasets[:5]:
    raw = rec.raw
    raw.load_data()
    raw.pick(raw.ch_names[:KEEP_CHANNELS])

    anns = raw.annotations

    for i, ann in enumerate(anns):
        desc = ann["description"]
        onset = ann["onset"]

        if desc == "OVTK_StimulationId_BaselineStart":
            label = 0
        elif desc in ["OVTK_GDF_Right", "OVTK_GDF_Tongue"]:
            label = 1
        else:
            continue

        # until next annotation
        tmax = anns[i + 1]["onset"] if i < len(anns) - 1 else raw.times[-1]

        if tmax - onset < WINDOW_SEC:
            continue

        segment = raw.copy().crop(tmin=onset, tmax=tmax, include_tmax=False)

        X, y = window_raw(segment, label)
        all_X.extend(X)
        all_y.extend(y)

X_DS003810 = np.array(all_X)
y_DS003810 = np.array(all_y)

print(X_DS003810.shape, np.bincount(y_DS003810))

Reading 0 ... 53874  =      0.000 ...   430.992 secs...
Sampling frequency of the instance is already 125.0, returning unmodified.
Sampling frequency of the instance is already 125.0, returning unmodified.
Sampling frequency of the instance is already 125.0, returning unmodified.
Sampling frequency of the instance is already 125.0, returning unmodified.
Sampling frequency of the instance is already 125.0, returning unmodified.
Sampling frequency of the instance is already 125.0, returning unmodified.
Sampling frequency of the instance is already 125.0, returning unmodified.
Sampling frequency of the instance is already 125.0, returning unmodified.
Sampling frequency of the instance is already 125.0, returning unmodified.
Sampling frequency of the instance is already 125.0, returning unmodified.
Sampling frequency of the instance is already 125.0, returning unmodified.
Sampling frequency of the instance is already 125.0, returning unmodified.
Sampling frequency of the instance is alread

# reclassification into rest vs main

In [17]:
print(X_DS006848.shape, np.bincount(y_DS006848))
print(X_DS003810.shape, np.bincount(y_DS003810))

(69911, 15, 31) [ 6004 63907]
(2641, 15, 31) [   0 2641]


# concatenation

In [18]:
# combine
X_all = np.concatenate([X_DS006848, X_DS003810], axis=0)
y_all = np.concatenate([y_DS006848, y_DS003810], axis=0)

# split into classes
mask_0 = (y_all == 0)
mask_1 = (y_all == 1)

X_final_0 = X_all[mask_0]
y_final_0 = y_all[mask_0]

X_final_1 = X_all[mask_1]
y_final_1 = y_all[mask_1]

print("class 0:", X_final_0.shape, np.bincount(y_final_0))
print("class 1:", X_final_1.shape, np.bincount(y_final_1))

class 0: (6004, 15, 31) [6004]
class 1: (66548, 15, 31) [    0 66548]


In [19]:
X_final = np.concatenate([X_final_0, X_final_1], axis=0)
y_final = np.concatenate([y_final_0, y_final_1], axis=0)

print(X_final.shape, np.bincount(y_final))

(72552, 15, 31) [ 6004 66548]


# train test split and features

In [50]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_final,
    y_final,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=y_final
)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(58041, 15, 31) (14511, 15, 31)
0.9172481521682948 0.9172352008820894


In [51]:
X_train_feat, feature_info_train = extract_features_fast(X_train, 125)
X_test_feat, feature_info_test  = extract_features_fast(X_test, 125)

print(X_train_feat.shape)

/Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/scipy/signal/_spectral_py.py:600: UserWarning: nperseg = 250 is greater than input length  = 31, using nperseg = 31
  freqs, _, Pxy = _spectral_helper(x, y, fs, window, nperseg, noverlap,


(58041, 120)


# model

In [52]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

model = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    # scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    n_jobs=-1
)

model.fit(X_train_feat, y_train)

preds = model.predict(X_test_feat)
probs = model.predict_proba(X_test_feat)[:,1]

print(classification_report(y_test, preds))
print("ROC AUC:", roc_auc_score(y_test, probs))

              precision    recall  f1-score   support

           0       0.90      0.19      0.32      1201
           1       0.93      1.00      0.96     13310

    accuracy                           0.93     14511
   macro avg       0.92      0.59      0.64     14511
weighted avg       0.93      0.93      0.91     14511

ROC AUC: 0.9086672701373949


# feature importances

In [53]:
importances = model.feature_importances_

start = 0

bp_imp = importances[start:start + feature_info_train["bandpower"]]
start += feature_info_train["bandpower"]

ent_imp = importances[start:start + feature_info_train["entropy"]]
start += feature_info_train["entropy"]

conn_imp = importances[start:start + feature_info_train["connectivity"]]
start += feature_info_train["connectivity"]

hjorth_imp = importances[start:start + feature_info_train["hjorth"]]

KeyError: 'connectivity'

In [47]:
group_importance = {
    "bandpower": bp_imp.sum(),
    "entropy": ent_imp.sum(),
    "connectivity": conn_imp.sum(),
    "hjorth": hjorth_imp.sum()
}

total = sum(group_importance.values())

group_importance = {k: v / total for k, v in group_importance.items()}

print(group_importance)

{'bandpower': np.float32(0.13471663), 'entropy': np.float32(0.10559406), 'connectivity': np.float32(0.46298644), 'hjorth': np.float32(0.29670292)}


In [48]:
top_conn = np.argsort(conn_imp)[::-1][:20]
print("Top connectivity features:", top_conn)
print("Their importance:", conn_imp[top_conn])

Top connectivity features: [404 515 400 479 447 505 510 368 414 509 252 503 342 399 288 482 519 499
 446 470]
Their importance: [0.04419197 0.01781332 0.01500762 0.01463094 0.01385368 0.01278459
 0.01160234 0.01155466 0.01129437 0.01109424 0.01033349 0.01031842
 0.01010736 0.00981515 0.00953538 0.00947213 0.00904982 0.00882065
 0.00838273 0.00813295]


# test model on a separate part of the dataset

In [60]:
import re
import numpy as np
from eegdash.dataset import DS006848

dataset = DS006848(cache_dir="./data")

def get_subject_id(raw):
    fname = str(raw.filenames[0]).lower()
    m = re.search(r"sub-(\d+)", fname)
    return m.group(1) if m else None


subject_to_recs = {}

for rec in dataset.datasets:
    raw = rec.raw

    # important: avoid re-downloading heavy data
    # only load metadata if possible
    raw.load_data()

    sid = get_subject_id(raw)
    if sid is None:
        continue

    subject_to_recs.setdefault(sid, []).append(rec)

subjects = sorted(subject_to_recs.keys())

print("Found subjects:", subjects)
print("Total:", len(subjects))

[04/20/26 19:58:51] WARNING  File not found on S3, skipping:                                      ]8;id=181018;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/downloader.py\downloader.py]8;;\:]8;id=499588;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/downloader.py#146\146]8;;\
                             s3://openneuro.org/ds006848/sub-021/eeg/sub-021_task-rest_eeg.dat                     

[04/20/26 19:58:53] WARNING  Companion file .dat not found on S3 for                                    ]8;id=617463;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/dataset/base.py\base.py]8;;\:]8;id=451142;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/dataset/base.py#483\483]8;;\
                             s3://openneuro.org/ds006848/sub-021/eeg/sub-021_task-rest_eeg.vhdr                    

Reading 0 ... 753799  =      0.000 ...   753.799 secs...


[04/20/26 19:58:57] WARNING  File not found on S3, skipping:                                      ]8;id=694586;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/downloader.py\downloader.py]8;;\:]8;id=583959;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/downloader.py#146\146]8;;\
                             s3://openneuro.org/ds006848/sub-021/eeg/sub-021_task-verbalwm_eeg.da                  
                             t                                                                                     

[04/20/26 19:58:59] WARNING  Companion file .dat not found on S3 for                                    ]8;id=452549;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/dataset/base.py\base.py]8;;\:]8;id=366072;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/dataset/base.py#483\483]8;;\
                             s3://openneuro.org/ds006848/sub-021/eeg/sub-021_task-verbalwm_eeg.vhdr                

...

Reading 0 ... 728279  =      0.000 ...   728.279 secs...


[04/20/26 20:13:24] WARNING  File not found on S3, skipping:                                      ]8;id=397129;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/downloader.py\downloader.py]8;;\:]8;id=361752;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/downloader.py#146\146]8;;\
                             s3://openneuro.org/ds006848/sub-020/eeg/sub-020_task-verbalwm_eeg.da                  
                             t                                                                                     

[04/20/26 20:13:27] WARNING  Companion file .dat not found on S3 for                                    ]8;id=447169;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/dataset/base.py\base.py]8;;\:]8;id=897019;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/dataset/base.py#483\483]8;;\
                             s3://openneuro.org/ds006848/sub-020/eeg/sub-020_task-verbalwm_eeg.vhdr                

Reading 0 ... 4745579  =      0.000 ...  4745.579 secs...


FSTimeoutError: 

In [ ]:
n = len(subjects)
mid = n // 2

train_subjects = set(subjects[:mid])
test_subjects  = set(subjects[mid:])

print("Train subjects:", train_subjects)
print("Test subjects:", test_subjects)

In [ ]:
all_X_train, all_y_train = [], []
all_X_test, all_y_test = [], []

for sid, recs in subject_to_recs.items():

    for rec in recs:
        raw = rec.raw
        raw.load_data()

        raw.pick(raw.ch_names[:KEEP_CHANNELS])

        name = str(raw.filenames[0]).lower()

        if "task-rest" in name:
            label = 0
        elif "task-verbalwm" in name:
            label = 1
        else:
            continue

        X, y = window_raw(raw, label)

        if sid in train_subjects:
            all_X_train.extend(X)
            all_y_train.extend(y)
        else:
            all_X_test.extend(X)
            all_y_test.extend(y)


X_train = np.array(all_X_train)
y_train = np.array(all_y_train)

X_test = np.array(all_X_test)
y_test = np.array(all_y_test)

print("TRAIN:", X_train.shape, np.bincount(y_train))
print("TEST :", X_test.shape, np.bincount(y_test))

In [59]:
import numpy as np
import re
from eegdash.dataset import DS006848

WINDOW_SEC = 0.25
KEEP_CHANNELS = 15

dataset = DS006848(cache_dir="./data")

def get_subject_id(raw):
    fname = str(raw.filenames[0]).lower()
    match = re.search(r"sub-(\d+)", fname)
    return match.group(1) if match else None


# ----------------------------
# 1. collect unique subjects
# ----------------------------
subject_to_recs = {}

for rec in dataset.datasets:
    raw = rec.raw
    raw.load_data()

    sid = get_subject_id(raw)
    if sid is None:
        continue

    if sid not in subject_to_recs:
        subject_to_recs[sid] = []

    subject_to_recs[sid].append(rec)


# ----------------------------
# 2. pick 5 subjects explicitly
# ----------------------------
selected_subjects = list(subject_to_recs.keys())[:5]

print("Selected subjects:", selected_subjects)


# ----------------------------
# 3. build dataset (subject-safe)
# ----------------------------
all_X = []
all_y = []

for sid in selected_subjects:
    for rec in subject_to_recs[sid]:

        raw = rec.raw
        raw.load_data()
        raw.pick(raw.ch_names[:KEEP_CHANNELS])

        name = str(raw.filenames[0]).lower()

        if "task-rest" in name:
            label = 0
        elif "task-verbalwm" in name:
            label = 1
        else:
            continue

        X, y = window_raw(raw, label)

        all_X.extend(X)
        all_y.extend(y)

X_DS006848_new = np.array(all_X)
y_DS006848_new = np.array(all_y)

print(X_DS006848_new.shape, np.bincount(y_DS006848_new))

[04/20/26 18:45:25] WARNING  File not found on S3, skipping:                                      ]8;id=389983;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/downloader.py\downloader.py]8;;\:]8;id=939309;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/downloader.py#146\146]8;;\
                             s3://openneuro.org/ds006848/sub-021/eeg/sub-021_task-rest_eeg.dat                     

[04/20/26 18:45:27] WARNING  Companion file .dat not found on S3 for                                    ]8;id=921411;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/dataset/base.py\base.py]8;;\:]8;id=214868;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/dataset/base.py#483\483]8;;\
                             s3://openneuro.org/ds006848/sub-021/eeg/sub-021_task-rest_eeg.vhdr                    

Reading 0 ... 753799  =      0.000 ...   753.799 secs...


[04/20/26 18:45:31] WARNING  File not found on S3, skipping:                                      ]8;id=152872;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/downloader.py\downloader.py]8;;\:]8;id=990204;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/downloader.py#146\146]8;;\
                             s3://openneuro.org/ds006848/sub-021/eeg/sub-021_task-verbalwm_eeg.da                  
                             t                                                                                     

[04/20/26 18:45:33] WARNING  Companion file .dat not found on S3 for                                    ]8;id=132228;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/dataset/base.py\base.py]8;;\:]8;id=381805;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/dataset/base.py#483\483]8;;\
                             s3://openneuro.org/ds006848/sub-021/eeg/sub-021_task-verbalwm_eeg.vhdr                

...

Reading 0 ... 4652939  =      0.000 ...  4652.939 secs...


[04/20/26 19:16:34] WARNING  File not found on S3, skipping:                                      ]8;id=939218;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/downloader.py\downloader.py]8;;\:]8;id=652699;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/downloader.py#146\146]8;;\
                             s3://openneuro.org/ds006848/sub-018/eeg/sub-018_task-rest_eeg.dat                     

[04/20/26 19:16:37] WARNING  Companion file .dat not found on S3 for                                    ]8;id=507027;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/dataset/base.py\base.py]8;;\:]8;id=742771;file:///Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/eegdash/dataset/base.py#483\483]8;;\
                             s3://openneuro.org/ds006848/sub-018/eeg/sub-018_task-rest_eeg.vhdr                    

Reading 0 ... 734279  =      0.000 ...   734.279 secs...


OSError: [Errno 28] No space left on device

## testing

In [55]:
X_feat, _ = extract_features_fast(X_DS006848_new, 125)

preds = model.predict(X_feat)
probs = model.predict_proba(X_feat)[:,1]

print(classification_report(y_DS006848_new, preds))
print("ROC AUC:", roc_auc_score(y_DS006848_new, probs))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00      8874
           1       0.82      1.00      0.90     41826

    accuracy                           0.82     50700
   macro avg       0.41      0.50      0.45     50700
weighted avg       0.68      0.82      0.75     50700

ROC AUC: 0.5679629251898953


/Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p